# AP01 - Aggregatfunktionen

In [4]:
# Libraries
import pandas as pd
from sqlalchemy import create_engine, text
from pathlib import Path

# Current working directory
print('Arbeitsverzeichnis:', Path.cwd())

#NICHTS ANPASSEN: Imports und Path.cwd()
#ERKENNTNIS: Relative Dateipfade beziehen sich auf das Arbeitsverzeichnis des Notebooks

Arbeitsverzeichnis: /workspaces/DS_HS2026_LNW/DS_HS2026_LNW_I_Examples/AP01/AP01 Mock Exam


In [5]:
# Set up database connection
user = 'pgadmin'                                                                #Benutzername für PostgreSQL
password = 'geheim'                                                             #Passwort des Datenbankbenutzers
host = 'localhost'                                                              #Adresse des PostgreSQL-Servers
port = '5432'                                                                   #Port, über den PostgreSQL erreichbar ist
database = 'postgres'                                                           #Name der Datenbank, mit der verbunden wird

# Create Connection URL
db_connection_url = (
    f'postgresql://{user}:{password}@{host}:{port}/{database}'                  #Verbindungsadresse aus den oben definierten Angaben zusammensetzen
)                                                                               

# Create SQLAlchemy Engine
engine = create_engine(db_connection_url)                                       #Engine erstellen; Verbindung wird erst bei Bedarf geöffnet

# Test database connection
with engine.connect() as connection:                                            #Verbindung öffnen und nach dem Block automatisch schliessen
    result = connection.execute(text('SELECT current_database()'))              #Namen der aktuellen Datenbank abfragen
    print('Verbundene Datenbank:', result.scalar())                             #Einzelnen Ergebniswert auslesen und anzeigen

# Dispose the engine
engine.dispose()                                                                #Verbindungen im Pool schliessen und Ressourcen freigeben

Verbundene Datenbank: postgres


## Rules

### Welche Funktion für welche Frage?

| Funktion | Bedeutung |
|---|---|
| `COUNT(*)` | Anzahl aller Zeilen |
| `COUNT(spalte)` | Anzahl vorhandener Werte ohne NULL |
| `COUNT(DISTINCT spalte)` | Anzahl unterschiedlicher Werte ohne NULL |
| `SUM(spalte)` | Summe der vorhandenen Zahlen |
| `AVG(spalte)` | Durchschnitt der vorhandenen Zahlen |
| `MIN(spalte)` | Kleinster vorhandener Wert |
| `MAX(spalte)` | Grösster vorhandener Wert |

- `SUM` addiert Werte, `COUNT` zählt Zeilen oder vorhandene Werte
- `MIN` und `MAX` funktionieren auch mit Datumswerten, frühestes bzw. spätestes Datum

### Gesamtergebnis oder Ergebnis pro Gruppe?
- Ohne `GROUP BY` wird über alle nach dem Filtern verbleibenden Zeilen gerechnet
- Mit `GROUP BY make` entsteht ein Ergebnis pro Marke
- Mit `GROUP BY make, model` entsteht ein Ergebnis pro Kombination aus Marke und Modell
- Nicht aggregierte SELECT-Spalten grundsätzlich ebenfalls in `GROUP BY` aufnehmen
- PostgreSQL erlaubt bestimmte Ausnahmen, bspw. beim Gruppieren nach dem Primärschlüssel (das vollständige Mitgruppieren ist die einfache Prüfungsregel)
- Aggregation fasst Einzelzeilen zusammen, Details einzelner Datensätze gehen dabei verloren

### WHERE oder HAVING?
- `WHERE` filtert einzelne Zeilen vor der Aggregation, bspw. `WHERE price > 2000`
- `HAVING` filtert das berechnete Gruppenergebnis, bspw. `HAVING AVG(price) > 3000`
- Aggregatfunktionen gehören nicht direkt in `WHERE, für `COUNT(*) > 10` auf Gruppenebene `HAVING` verwenden
- Beide sind kombinierbar, zuerst passende Fahrräder auswählen, danach nur ausreichend grosse Gruppen behalten
- Bei gemischten Bedingungen mit `AND` und `OR` Klammern setzen, `AND` bindet stärker als `OR`

### NULL richtig behandeln
- `COUNT(*)` zählt auch Zeilen mit fehlenden Werten
- `COUNT(spalte)`, `COUNT(DISTINCT spalte)`, `SUM`, `AVG`, `MIN` und `MAX` ignorieren NULL
- 0 ist ein vorhandener Wert und wird berücksichtigt, auch beim Durchschnitt
- `AVG` teilt durch die Anzahl bekannter Werte, nicht durch sämtliche Zeilen
- Bspw. ergeben 100, 200 und NULL einen Durchschnitt von 150

### Runden und Datentypen
- In PostgreSQL benötigt `ROUND(wert, 2)` einen Wert vom Typ `NUMERIC`
- Bei Bedarf umwandeln, bspw. `ROUND(CAST(AVG(price) AS NUMERIC), 2)`
- `CAST` wandelt den Datentyp um, `ROUND` rundet den Wert
- `AS average_price` benennt die Ergebnisspalte

### Aliase und Sortierung
- SELECT-Aliase dürfen in `ORDER BY` verwendet werden, bspw. `ORDER BY average_price DESC`
- In `WHERE` und `HAVING` den ursprünglichen Ausdruck verwenden, bspw. `HAVING AVG(price) > 3000`
- `ASC` sortiert aufsteigend, `DESC` absteigend
- Ohne `ORDER BY` ist keine bestimmte Reihenfolge garantiert, auch nicht mit `LIMIT` oder `OFFSET`
- Spalten und Ausdrücke mit Kommas trennen, kein Komma direkt vor `FROM`

### Auswertung nach Jahr
- `EXTRACT(YEAR FROM servicedate)` liest das Jahr aus einem Datum
- `CAST(EXTRACT(YEAR FROM servicedate) AS INTEGER)` gibt es als Ganzzahl aus
- Für Kennzahlen pro Jahr nach diesem Jahresausdruck gruppieren

### Unterschied zu Fensterfunktionen
- `AVG(price)` fasst die betrachteten Zeilen zu einem Durchschnitt zusammen
- `AVG(price) OVER ()` behält jede Einzelzeile und ergänzt den Gesamtdurchschnitt als zusätzliche Spalte

### Reihenfolge einer typischen Abfrage
- `SELECT → FROM → WHERE → GROUP BY → HAVING → ORDER BY → LIMIT → OFFSET`
- Merksatz: Erst Zeilen filtern, dann Gruppen bilden und Kennzahlen berechnen, danach Gruppen filtern und Ergebnisse sortieren

### Ergänzende Prüfungsfallen
- Ohne bekannte Werte liefert `COUNT` 0, `SUM`, `AVG`, `MIN` und `MAX` liefern NULL
- Ohne `GROUP BY` entsteht auch bei leerer Auswahl eine Ergebniszeile, mit `GROUP BY` entstehen bei leerer Auswahl keine Gruppen
- Für die Auswahl nach Kennzahlen grundsätzlich ungerundete Werte prüfen und erst die Ausgabe runden
- `MIN(price)` liefert den kleinsten Preis, nicht automatisch das dazugehörige Fahrrad
- `GROUP BY` vereinheitlicht keine Schreibweisen — bspw. Standard und standard bilden getrennte Gruppen

## Queries

### Determine the total number of bicycles. Name the result bicycle_count
- `COUNT(*)` zählt alle Zeilen der Tabelle, ohne `GROUP BY` entsteht eine einzige Gesamtanzahl
- `AS bicycle_count` benennt die Ergebnisspalte

In [6]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT COUNT(*) AS bicycle_count        --Alle Zeilen zählen und das Ergebnis bicycle_count nennen
    FROM bicycles;                          --Ohne GROUP BY entsteht eine Gesamtanzahl für die Tabelle
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycle_count
0,2000


### Determine the total number of bicycles and the number of bicycles with a known price. Name the results bicycle_count and known_price_count
- `COUNT(*)` zählt alle Fahrräder, `COUNT(price)` nur jene mit vorhandenem Preis
- `NULL` wird bei `COUNT(price)` ignoriert, ein Preis von 0 wird mitgezählt

In [7]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT COUNT(*) AS bicycle_count,               --Alle Zeilen zählen, auch solche mit fehlendem Preis
           COUNT(price) AS known_price_count        --Nur Zeilen mit vorhandenem Preis zählen, ein Preis von 0 zählt mit
    FROM bicycles;                                  --Beide Anzahlen über die gesamte Tabelle berechnen
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycle_count,known_price_count
0,2000,2000


### Determine the number of distinct bicycle makes. Name the result make_count
- `COUNT(DISTINCT make)` zählt jede unterschiedliche Fahrradmarke nur einmal, fehlende Marken werden nicht mitgezählt
- `AS make_count` benennt die Ergebnisspalte

In [8]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT COUNT(DISTINCT make) AS make_count    --Unterschiedliche Marken zählen, Duplikate und NULL werden nicht mitgezählt
    FROM bicycles;                               --Eine Gesamtanzahl über die gesamte Tabelle berechnen
""", con=engine)

engine.dispose()
display(df_spickap01)

,make_count
0,10


### Determine the number of bicycles per status. Display status and bicycle_count, sorted by bicycle_count in descending order
- `GROUP BY status` bildet eine Gruppe pro Status, `COUNT(*)` zählt die Fahrräder je Gruppe
- `ORDER BY bicycle_count DESC` zeigt den Status mit den meisten Fahrrädern zuerst

In [7]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT status,                          --Status jeder Gruppe anzeigen
           COUNT(*) AS bicycle_count        --Anzahl Fahrräder innerhalb jeder Statusgruppe zählen
    FROM bicycles                           --Daten aus der Tabelle bicycles abrufen
    GROUP BY status                         --Fahrräder mit demselben Status zu einer Gruppe zusammenfassen
    ORDER BY bicycle_count DESC;            --Gruppen mit den meisten Fahrrädern zuerst anzeigen
""", con=engine)

engine.dispose()
display(df_spickap01)

,status,bicycle_count
0,Sold,684
1,Available,672
2,In Maintenance,644


### Determine the number of distinct bicycle models per make. Display make and model_count, sorted by model_count in descending order
- `GROUP BY make` bildet eine Gruppe pro Marke, `COUNT(DISTINCT model)` zählt unterschiedliche Modelle je Marke ohne NULL
- `ORDER BY model_count DESC` zeigt die Marke mit den meisten unterschiedlichen Modellen zuerst

In [9]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT make,                                 --Fahrradmarke jeder Gruppe anzeigen
           COUNT(DISTINCT model) AS model_count  --Unterschiedliche Modelle pro Marke zählen; NULL nicht mitzählen
    FROM bicycles                                --Daten aus der Tabelle bicycles abrufen
    GROUP BY make                                --Fahrräder nach Marke gruppieren
    ORDER BY model_count DESC;                   --Marken mit den meisten unterschiedlichen Modellen zuerst anzeigen
""", con=engine)

engine.dispose()
display(df_spickap01)

,make,model_count
0,Bianchi,3
1,Cannondale,3
2,Cervelo,3
3,Cube,3
4,Giant,3
5,Merida,3
6,Santa Cruz,3
7,Scott,3
8,Specialized,3
9,Trek,3


### Determine the average bicycle price for each make. Display make and average_price, rounded to two decimal places and sorted by average_price in descending order, with missing averages last
- `GROUP BY make` bildet eine Gruppe pro Marke, `AVG(price)` berechnet den Durchschnitt ohne NULL und `ROUND(CAST(... AS NUMERIC), 2)` rundet auf zwei Dezimalstellen
- `ORDER BY average_price DESC NULLS LAST` zeigt den höchsten Durchschnitt zuerst, Marken ohne bekannte Preise stehen am Ende

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT make,                                                    --Fahrradmarke jeder Gruppe anzeigen
           ROUND(CAST(AVG(price) AS NUMERIC), 2) AS average_price   --Durchschnitt berechnen, in NUMERIC umwandeln und auf zwei Stellen runden
    FROM bicycles                                                   --Daten aus der Tabelle bicycles abrufen
    GROUP BY make                                                   --Durchschnitt für jede Marke separat berechnen, AVG ignoriert NULL
    ORDER BY average_price DESC NULLS LAST;                         --Höchsten Durchschnitt zuerst anzeigen, fehlende Durchschnitte zuletzt
""", con=engine)

engine.dispose()
display(df_spickap01)

,make,average_price
0,Cube,4603.98
1,Bianchi,4556.95
2,Santa Cruz,4499.56
3,Specialized,4477.58
4,Trek,4417.79
5,Cannondale,4340.02
6,Cervelo,4313.50
7,Giant,4225.80
8,Merida,4218.95
9,Scott,4193.24


### Determine the average bicycle price for each combination of make and model. Display make, model and average_price, rounded to two decimal places. Sort by make and then by average_price, both in descending order
- `GROUP BY make, model` bildet eine Gruppe pro Kombination aus Marke und Modell, `AVG(price)` berechnet deren Durchschnitt und `ROUND(CAST(... AS NUMERIC), 2)` rundet auf zwei Dezimalstellen
- `ORDER BY make DESC, average_price DESC` sortiert Marken von Z nach A und innerhalb jeder Marke den höchsten Durchschnitt zuerst

In [10]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT make,                                                    --Fahrradmarke auswählen
           model,                                                   --Fahrradmodell auswählen
           ROUND(CAST(AVG(price) AS NUMERIC), 2) AS average_price   --Durchschnittspreis je Gruppe auf zwei Dezimalstellen runden
    FROM bicycles                                                   --Daten aus der Tabelle bicycles abrufen
    GROUP BY make, model                                            --Für jede Kombination aus Marke und Modell eine Gruppe bilden
    ORDER BY make DESC,                                             --Zuerst nach Marke absteigend sortieren
             average_price DESC;                                    --Innerhalb derselben Marke nach Durchschnittspreis absteigend sortieren
""", con=engine)

engine.dispose()
display(df_spickap01)

,make,model,average_price
0,Trek,Marlin,4666.03
1,Trek,Domane,4466.53
2,Trek,Emonda,4119.89
3,Specialized,Rockhopper,4742.53
4,Specialized,Roubaix,4504.83
5,Specialized,Allez,4128.30
6,Scott,Aspect,4297.69
7,Scott,Addict,4178.61
8,Scott,Spark,4091.72
9,Santa Cruz,Bronson,4747.68


### Determine how many bicycles per make have a price between 2000 and 4000, including both limits. Display make and bicycle_count, sorted by bicycle_count in descending order
- `WHERE price BETWEEN 2000 AND 4000` filtert vor dem Gruppieren auf Preise inkl. beider Grenzen, `GROUP BY make` und `COUNT(*)` zählen die passenden Fahrräder je Marke
- `ORDER BY bicycle_count DESC` zeigt die Marke mit den meisten passenden Fahrrädern zuerst

In [12]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT make,                            --Fahrradmarke jeder Gruppe anzeigen
           COUNT(*) AS bicycle_count        --Anzahl Fahrräder pro Marke nach der Filterung zählen
    FROM bicycles                           --Daten aus der Tabelle bicycles abrufen
    WHERE price BETWEEN 2000 AND 4000       --Vor dem Gruppieren auf Preise von 2000 bis 4000 inklusive filtern
    GROUP BY make                           --Verbleibende Fahrräder nach Marke gruppieren
    ORDER BY bicycle_count DESC;            --Marken mit den meisten passenden Fahrrädern zuerst anzeigen
""", con=engine)

engine.dispose()
display(df_spickap01)

,make,bicycle_count
0,Cannondale,66
1,Giant,64
2,Bianchi,57
3,Specialized,57
4,Santa Cruz,54
5,Cervelo,54
6,Cube,52
7,Merida,50
8,Scott,45
9,Trek,42


### Identify bicycle makes with an average price greater than 3000. Display make and average_price, rounded to two decimal places and sorted by average_price in descending order
- `GROUP BY make` bildet Gruppen pro Marke, `HAVING AVG(price) > 3000` behält nur Gruppen mit einem ungerundeten Durchschnitt über 3000
- `ROUND(CAST(... AS NUMERIC), 2)` rundet die Ausgabe auf zwei Dezimalstellen, `ORDER BY average_price DESC` zeigt den höchsten Durchschnitt zuerst

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT make,                                                        --Fahrradmarke jeder Gruppe anzeigen
           ROUND(CAST(AVG(price) AS NUMERIC), 2) AS average_price       --Durchschnittspreis berechnen und auf zwei Dezimalstellen runden
    FROM bicycles                                                       --Daten aus der Tabelle bicycles abrufen
    GROUP BY make                                                       --Fahrräder nach Marke gruppieren
    HAVING AVG(price) > 3000                                            --Gruppen nach der Aggregation filtern, ungerundeten Durchschnitt prüfen
    ORDER BY average_price DESC;                                        --Höchsten Durchschnitt zuerst anzeigen
""", con=engine)

engine.dispose()
display(df_spickap01)

,make,average_price
0,Cube,4603.98
1,Bianchi,4556.95
2,Santa Cruz,4499.56
3,Specialized,4477.58
4,Trek,4417.79
5,Cannondale,4340.02
6,Cervelo,4313.50
7,Giant,4225.80
8,Merida,4218.95
9,Scott,4193.24


### Identify bicycle makes with more than 10 bicycles priced between 2000 and 4000, including both price limits. Display make and bicycle_count, sorted by bicycle_count in descending order
- `WHERE price BETWEEN 2000 AND 4000` filtert einzelne Fahrräder inkl. beider Grenzen, anschliessend zählt `COUNT(*)` je Marke
- `HAVING COUNT(*) > 10` behält nur Marken mit mehr als zehn passenden Fahrrädern, `ORDER BY bicycle_count DESC` zeigt die grösste Anzahl zuerst

In [13]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT make,                                --Fahrradmarke jeder Gruppe anzeigen
           COUNT(*) AS bicycle_count            --Anzahl der Fahrräder pro Marke nach der Preisfilterung zählen
    FROM bicycles                               --Daten aus der Tabelle bicycles abrufen
    WHERE price BETWEEN 2000 AND 4000           --Einzelne Zeilen vor der Gruppierung nach Preis filtern
    GROUP BY make                               --Verbleibende Fahrräder nach Marke gruppieren
    HAVING COUNT(*) > 10                        --Nur Gruppen mit mehr als zehn passenden Fahrrädern behalten
    ORDER BY bicycle_count DESC;                --Gruppen mit den meisten passenden Fahrrädern zuerst anzeigen
""", con=engine)

engine.dispose()
display(df_spickap01)

,make,bicycle_count
0,Cannondale,66
1,Giant,64
2,Bianchi,57
3,Specialized,57
4,Santa Cruz,54
5,Cervelo,54
6,Cube,52
7,Merida,50
8,Scott,45
9,Trek,42


### Determine the minimum, maximum and average bicycle price. Name the results min_price, max_price and average_price, and round all three values to two decimal places
- `MIN(price)`, `MAX(price)` und `AVG(price)` berechnen den kleinsten, grössten und durchschnittlichen Preis über die gesamte Tabelle, NULL wird ignoriert
- `ROUND(CAST(... AS NUMERIC), 2)` rundet jeden Ergebniswert auf zwei Dezimalstellen

In [13]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT ROUND(CAST(MIN(price) AS NUMERIC), 2) AS min_price,      --Kleinsten vorhandenen Preis ermitteln und auf zwei Stellen runden
           ROUND(CAST(MAX(price) AS NUMERIC), 2) AS max_price,      --Grössten vorhandenen Preis ermitteln und auf zwei Stellen runden
           ROUND(CAST(AVG(price) AS NUMERIC), 2) AS average_price   --Durchschnitt der vorhandenen Preise auf zwei Stellen runden
    FROM bicycles;                                                  --Ohne GROUP BY über die gesamte Tabelle rechnen; NULL wird ignoriert
""", con=engine)

engine.dispose()
display(df_spickap01)

,min_price,max_price,average_price
0,800.26,7995.9,4382.23


### Determine the total sales revenue and the number of sales. Name the results total_sales_amount and sales_count, and round the total sales revenue to two decimal places
- `SUM(saleprice)` addiert die Verkaufspreise ohne NULL, `COUNT(*)` zählt alle Verkäufe inkl. jener ohne bekannten Verkaufspreis
- `ROUND(CAST(SUM(saleprice) AS NUMERIC), 2)` rundet den Gesamtumsatz auf zwei Dezimalstellen

In [14]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT ROUND(CAST(SUM(saleprice) AS NUMERIC), 2) AS total_sales_amount,     --Verkaufspreise addieren und auf zwei Stellen runden, NULL ignorieren
           COUNT(*) AS sales_count                                              --Alle Verkaufsdatensätze zählen
    FROM sales;                                                                 --Über die gesamte Verkaufstabelle rechnen, kein GROUP BY nötig
""", con=engine)

engine.dispose()
display(df_spickap01)

,total_sales_amount,sales_count
0,2158850.0,684


### Determine the total bicycle list value per make. Display make and total_value, include only makes with a total value greater than 100000, and sort by total_value in descending order. Round total_value to two decimal places
- `GROUP BY make` bildet Gruppen pro Marke, `SUM(price)` addiert die Listenpreise je Marke ohne NULL
- `HAVING SUM(price) > 100000` filtert nach dem ungerundeten Gesamtwert, die Ausgabe wird gerundet und absteigend sortiert

In [15]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT make,                                                    --Fahrradmarke auswählen
           ROUND(CAST(SUM(price) AS NUMERIC), 2) AS total_value     --Listenpreise je Marke addieren und auf zwei Dezimalstellen runden
    FROM bicycles                                                   --Fahrraddaten verwenden
    GROUP BY make                                                   --Eine Gruppe pro Marke bilden
    HAVING SUM(price) > 100000                                      --Nur Marken mit ungerundetem Gesamtwert über 100000 behalten
    ORDER BY total_value DESC;                                      --Höchsten Gesamtwert zuerst anzeigen
""", con=engine)

engine.dispose()
display(df_spickap01)

,make,total_value
0,Santa Cruz,962906.23
1,Giant,921223.85
2,Specialized,899993.44
3,Cube,888568.72
4,Cannondale,876684.87
5,Bianchi,870378.23
6,Merida,864883.90
7,Scott,838647.94
8,Cervelo,823878.02
9,Trek,817290.36
